# **8주차: Natural Language Processing**

In [ ]:
!pip install torch
!pip install datasets

### **데이터 불러오기 및 전처리**

In [ ]:
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from konlpy.tag import Okt

In [ ]:
# 데이터셋 불러오기
dataset = load_dataset("Blpeng/nsmc")

# 토크나이저 선언
okt = Okt()   # Open Korean Text Processor: 한국어 텍스트를 토큰화하는 형태소 분석기


# 텍스트 토큰화 함수
def tokenize(sents):      # sent : 입력 문장 리스트
    tokenized_sents = []
    for sent in tqdm(sents):
        tokenized_sent = [token for token in okt.morphs(sent)]  # 형태소 단위로 토큰화
        tokenized_sents.append(tokenized_sent)
    return tokenized_sents # tokenized_sents (list): 토큰화된 문장 리스트


# 토큰화된 문장 리스트로부터 단어 사전 생성 함수
def make_vocab(tokenized_sents, max_size=50000):
    """
    Args:
        tokenized_sents (list): 토큰화된 문장 리스트
        max_size (int): 단어 사전 최대 크기 (기본값: 50,000)
    Returns:
        vocab (dict): 단어와 인덱스를 매핑한 사전
    """
    all_tokens = []
    for tokens in tokenized_sents:
        all_tokens += tokens        # 모든 문장의 토큰을 하나의 리스트로 합침
    counter = Counter(all_tokens)   # 토큰의 빈도수를 계산
    word_list = list(map(lambda x: x[0], counter.most_common(max_size-2))) # 상위 max_size-2개의 단어 선택

    vocab = {"[PAD]": 0, "[UNK]": 1}  # [PAD]: 패딩 토큰, [UNK]: 알 수 없는 단어 토큰
    for i, word in enumerate(word_list):
        vocab[word] = i + 2         # 인덱스 2부터 실제 단어 매핑
    return vocab

# 토큰을 인덱스로 변환하는 함수
def token_to_idx(tokenized_sents, vocab, seq_len=32):
    """
    토큰화된 문장을 정수 인덱스 시퀀스로 변환합니다.
    Args:
        tokenized_sents (list): 토큰화된 문장 리스트
        vocab (dict): 단어와 인덱스를 매핑한 사전
        seq_len (int): 문장 최대 길이 (기본값: 32)
    Returns:
        sent_idxs (list): 정수 인덱스 리스트
    """
    sent_idxs = []
    for sent in tqdm(tokenized_sents):
        if len(sent) > seq_len:
            sent = sent[:seq_len] # 문장이 seq_len보다 길면 자름
        elif len(sent) < seq_len: # 문장이 seq_len보다 짧으면 [PAD]로 채움
            sent = sent + ["[PAD]"] * (seq_len-len(sent))
        idxs = []
        for word in sent:         # 각 단어를 인덱스로 변환
            if word in vocab:
                idxs.append(vocab[word])
            else:
                idxs.append(vocab["[UNK]"]) # 사전에 없는 단어는 [UNK]로 처리
        sent_idxs.append(idxs)    # 변환된 인덱스를 리스트에 추가
    return sent_idxs

In [ ]:
# 결측(null) 데이터 제거
print("Remove null data progress...")
dataset = dataset.filter(lambda x: x["document"] is not None)

# 문장 토큰화
print("\nTokenize progress...")
X_train = tokenize(dataset["train"]["document"])
X_test = tokenize(dataset["test"]["document"])

# 단어 사전 생성
vocab = make_vocab(X_train, max_size=50000)

# 토큰을 정수 인덱스로 변환
print("\nToken to index progress...")
X_train = torch.LongTensor(token_to_idx(X_train, vocab, seq_len=32))
X_test = torch.LongTensor(token_to_idx(X_test, vocab, seq_len=32))

# 레이블 데이터를 정수형 텐서로 변환
y_train = torch.LongTensor(dataset["train"]["label"])
y_test = torch.LongTensor(dataset["test"]["label"])

### **LSTM 모델 생성**

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# LSTM 텍스트 분류 모델 정의
class MyModel(nn.Module):
    def __init__(self,
                 num_labels,      # 출력 클래스 수
                 vocab_size=50000, # 단어 사전 크기
                 embed_size=128,  # 임베딩 벡터 크기
                 hid_size=128,    # LSTM의 히든 레이어 크기
                 num_layers=2,    # LSTM 레이어 수
                 bidirectional=True, # 양방향 LSTM 사용 여부
                 p=0.2):          # 양방향 LSTM 사용 여부

        super().__init__()

        # 임베딩 레이어: 입력 토큰을 실수 벡터로 매핑
        self.embedding = nn.Sequential(
            nn.Embedding(vocab_size, embed_size),
            nn.Dropout(p=p))

        # LSTM 레이어: 시퀀스 데이터 처리
        self.layer = nn.LSTM(
            input_size=embed_size,  # 임베딩 크기
            hidden_size=hid_size,   # 은닉층 크기
            num_layers=num_layers,  # LSTM 레이어 수
            dropout=p,              # 드롭아웃 비율
            bidirectional=bidirectional, # 양방향 LSTM
            batch_first=True)       # 입력 데이터의 첫 번째 차원이 배치

        # 분류기 레이어: 마지막 히든 상태를 이용해 클래스 분류
        self.classifier = nn.Sequential(nn.Linear(hid_size*2 if bidirectional else hid_size, num_labels)) # 출력 크기 조정


    def forward(self, x): # 순전파 입력: 입력 텍스트의 정수 인덱스 시퀀스
        x = self.embedding(x)
        x = self.layer(x)[0][:,-1]
        x = self.classifier(x)
        return x          # 클래스별 로짓 값

# 모델 생성
model = MyModel(num_labels=2)

# 모델 구조 출력
print(model)

### **모델 학습**

In [ ]:
# 하이퍼파라미터 설정
epochs = 3
batch_size = 64
lr = 1e-4
weight_decay = 1e-2 # 가중치 감소율 (정규화 목적)

# 손실함수, 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# 데이터셋 및 데이터로더 생성
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 모델 학습
for epoch in range(1,epochs+1):
    train_loss, train_acc, test_loss, test_acc = [], [], [], []
    model.train()                              # 모델 학습 모드
    for data, label in train_dataloader:
        pred = model(data)
        loss = criterion(pred, label)
        train_loss.append(loss)
        train_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

        optimizer.zero_grad()                  # 파라미터의 누적된 기울기 초기화
        loss.backward()                        # 기울기 계산
        optimizer.step()                       # 누적된 기울기를 이용해 옵티마이저로 파라미터 업데이트

    model.eval()                               # 모델 평가 모드
    for data, label in test_dataloader:
        with torch.no_grad():                  # 자동 미분(autograd) 비활성화 (계산 속도 및 메모리 효율 개선)
            pred = model(data)
            loss = criterion(pred, label)
            test_loss.append(loss)
            test_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

    #손실 및 정확도 계산
    train_loss = torch.stack(train_loss).mean().item()
    train_acc = torch.stack(train_acc).mean().item()
    test_loss = torch.stack(test_loss).mean().item()
    test_acc = torch.stack(test_acc).mean().item()

    print(f"Epoch {epoch}/{epochs}")
    print(f"train loss: {round(train_loss, 4)}  train accuracy: {round(train_acc, 4)}")
    print(f"test loss: {round(test_loss, 4)}  test accuracy: {round(test_acc, 4)}\n")